# Driblab - Event Detection: ETL
**Step 1 of the project: load, inspect, and transform the raw data before touching a model.**

This notebook works through the Step 1 checklist from the kickoff guide: the event vocabulary, the events file, the tracking file, and how the two connect. We call this ETL because this step validates and transforms coordinate fields, not just explores them.

**Files used:**
- `dim_event_type.csv` - event type vocabulary (84 defined event types)
- `679026_events.json` - labelled events for match 679026 (Arsenal FC 2-0 FC Brentford)
- `679026_tracking_data.jsonl` - tracking data for the same match

**Structure of this notebook:**
0. Setup - load the data
1. Event Vocabulary - what event types exist vs what actually occurs
2. Events + Tracking - validate event coordinates and normalize tracking coordinates
3. Events File - time encoding, the `outcome` field, and `qualifiers`
4. Tracking File - data quality, visibility, and sampling rate
5. Cross-File Consistency - do events and tracking line up, and what assets do we have?
6. Summary - key takeaways for the team


## Setup — Load Data
**Goal:** load the event-type vocabulary, the labelled events for match 679026, and the corresponding tracking data, so we can explore them below.


In [1]:
# ── Cell 1: Load all data ─────────────────────────────────────────────────────
import json
import csv
import os
import re
from collections import Counter
import pandas as pd

DATA_DIR = os.path.expanduser('~/Documents/IE/MBDS/Term III/Driblab/data/raw')

# ── Load event type definitions ───────────────────────────────────────────────
DATA_DIR = os.path.expanduser('~/Documents/IE/MBDS/Term III/Driblab/data/raw')
with open(os.path.join(DATA_DIR, 'dim_event_type.csv')) as f:
    event_types = list(csv.DictReader(f))
df_event_types = pd.DataFrame(event_types)

# ── Load labelled events (match 679026) ───────────────────────────────────────
# Note: file has minor corruption, parsing object-by-object
with open(os.path.join(DATA_DIR, '679026_events.json'), errors='ignore') as f:
    content = f.read()
events_raw = []
for match in re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', content):
    try:
        events_raw.append(json.loads(match.group()))
    except:
        pass
df_events = pd.json_normalize(events_raw)

# ── Load tracking data (match 679026 — same match as events) ─────────────────
tracking_lines = open(os.path.join(DATA_DIR, '679026_tracking_data.jsonl')).readlines()
tracking_header = json.loads(tracking_lines[0])
tracking_frames = [json.loads(l) for l in tracking_lines[1:]]

print(f"✅ Event types loaded:   {len(df_event_types)} rows")
print(f"✅ Labelled events:      {len(df_events)} rows (679026)")
print(f"✅ Tracking frames:      {len(tracking_frames)} (679026)")
print(f"\nMatch info: {tracking_header['match_data']}")
print(f"\nEvent columns: {list(df_events.columns)}")

✅ Event types loaded:   84 rows
✅ Labelled events:      1424 rows (679026)
✅ Tracking frames:      59502 (679026)

Match info: {'season_data': {'name': 'ENG I 2025', 'id': 818671}, 'date': '2025-12-03', 'match_id': 679026, 'result': {'home': 2, 'away': 0}}

Event columns: ['match_id', 'period_id', 'min', 'sec', 'x', 'y', 'outcome', 'qualifiers', 'possession_id', 'xa', 'xg', 'xt', 'x_start', 'y_start', 'x_end', 'y_end', 'milisec', 'event.id', 'event.event_type_id', 'event.event_type_name', 'team.team_id', 'team.team_name', 'player.player_id', 'player.player_name']


In [2]:
# ── Cell 2: Coordinate range summary for event and tracking coordinate fields ──
from IPython.display import display

def coordinate_columns(df):
    patterns = r'\b(?:x|y|x_start|y_start|x_end|y_end|x_raw|y_raw|meter|metre|meters|metres)\b'
    return [c for c in df.columns if re.search(patterns, c, flags=re.IGNORECASE)]

events_coords = df_events.copy()
event_cols = coordinate_columns(events_coords)
events_coords[event_cols] = events_coords[event_cols].apply(pd.to_numeric, errors='coerce')

tracking_ball_rows = []
tracking_player_rows = []
for frame in tracking_frames:
    ball = frame.get('ball') or [None, None, None]
    tracking_ball_rows.append({
        'ball_x_raw_m': ball[0],
        'ball_y_raw_m': ball[1],
        'ball_z_raw_m': ball[2],
    })
    for team_id, players in (frame.get('data') or {}).items():
        for player in players:
            tracking_player_rows.append({
                'player_x_raw_m': player.get('x'),
                'player_y_raw_m': player.get('y'),
            })

df_tracking_ball = pd.DataFrame(tracking_ball_rows)
df_tracking_players = pd.DataFrame(tracking_player_rows)

for df in [df_tracking_ball, df_tracking_players]:
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

tracking_cols = sorted(set(df_tracking_ball.columns) | set(df_tracking_players.columns))

rows = []
for col in sorted(set(event_cols + tracking_cols)):
    if col in events_coords.columns:
        values = events_coords[col].dropna()
        rows.append({
            'dataset': 'events',
            'field': col,
            'unit': 'normalized 0-100' if col in ['x', 'y', 'x_start', 'y_start', 'x_end', 'y_end'] else 'numeric',
            'min': values.min() if not values.empty else pd.NA,
            'max': values.max() if not values.empty else pd.NA,
            'non_null_rows': int(values.count()),
        })
    if col in df_tracking_ball.columns:
        values = df_tracking_ball[col].dropna()
        rows.append({
            'dataset': 'tracking_ball',
            'field': col,
            'unit': 'meters',
            'min': values.min() if not values.empty else pd.NA,
            'max': values.max() if not values.empty else pd.NA,
            'non_null_rows': int(values.count()),
        })
    if col in df_tracking_players.columns:
        values = df_tracking_players[col].dropna()
        rows.append({
            'dataset': 'tracking_player',
            'field': col,
            'unit': 'meters',
            'min': values.min() if not values.empty else pd.NA,
            'max': values.max() if not values.empty else pd.NA,
            'non_null_rows': int(values.count()),
        })

df_coordinate_ranges = pd.DataFrame(rows)
print('=== Coordinate range summary for all coordinate-like fields ===')
display(df_coordinate_ranges[['dataset','field','unit','min','max','non_null_rows']].sort_values(['dataset','field']).reset_index(drop=True))

=== Coordinate range summary for all coordinate-like fields ===


,dataset,field,unit,min,max,non_null_rows
0,events,x,normalized 0-100,0.00,100.000,1424
1,events,x_end,normalized 0-100,1.00,100.000,834
2,events,x_start,normalized 0-100,0.00,100.000,1133
3,events,y,normalized 0-100,0.00,100.000,1424
4,events,y_end,normalized 0-100,2.00,95.000,834
5,events,y_start,normalized 0-100,0.00,100.000,1133
6,tracking_ball,ball_x_raw_m,meters,-43.53,265.540,31630
7,tracking_ball,ball_y_raw_m,meters,-6.23,205.790,31630
8,tracking_ball,ball_z_raw_m,meters,0.00,10.512,31630
9,tracking_player,player_x_raw_m,meters,-1.77,107.130,1309044


**Note:** the events JSON file has minor corruption and is parsed object-by-object (1424 of the events recovered successfully). All three files load correctly and concern the same match (Arsenal FC 2-0 FC Brentford, 2025-12-03).


In [3]:
tracking_lines = open(os.path.join(DATA_DIR, '679026_tracking_data.jsonl')).readlines()
print(tracking_lines[0])

{"match_data": {"season_data": {"name": "ENG I 2025", "id": 818671}, "date": "2025-12-03", "match_id": 679026, "result": {"home": 2, "away": 0}}, "teams_data": {"home": {"name": "Arsenal FC", "id": 1928}, "away": {"name": "FC Brentford", "id": 8561}}, "players_data": {"1928": {"195773": {"number": 8, "name": "Martin \u00d8degaard", "position": "MC"}, "257909": {"number": 1, "name": "David Raya", "position": "GK"}, "1158364": {"number": 41, "name": "Declan Rice", "position": "MC"}, "1158442": {"number": 4, "name": "Ben White", "position": "DR"}, "1158710": {"number": 23, "name": "Mikel Merino", "position": "FW"}, "1267742": {"number": 5, "name": "Piero Hincapie", "position": "DC"}, "1268049": {"number": 36, "name": "Martin Zubimendi", "position": "DMC"}, "1295026": {"number": 11, "name": "Martinelli", "position": "FWL"}, "1308153": {"number": 33, "name": "Riccardo Calafiori", "position": "DL"}, "1312824": {"number": 20, "name": "Noni Madueke", "position": "FWR"}, "1701729": {"number": 3

In [4]:
# Inspect the tracking JSONL structure
sample_frame = tracking_frames[0]



print("=== Top-level keys in one tracking frame ===")
print(list(sample_frame.keys()))

print("\n=== Ball keys ===")
print(sample_frame['ball'])

print("\n=== Team keys under frame['data'] ===")
print(list(sample_frame['data'].keys()))

print("\n=== Player keys for one player ===")
one_team = next(iter(sample_frame['data'].values()))
one_player = one_team[0]
print(list(one_player.keys()))

print("\n=== Tracking header keys ===")
print(list(tracking_header.keys()))

=== Top-level keys in one tracking frame ===
['data', 'match_clock', 'frame', 'Videotimestamp', 'period', 'ball', 'cam']

=== Ball keys ===
[None, None, None]

=== Team keys under frame['data'] ===
['8561', '1928']

=== Player keys for one player ===
['x', 'y', 'vis', 'id']

=== Tracking header keys ===
['match_data', 'teams_data', 'players_data', 'FPS']


In [5]:
# ── Cell 2: Event type distribution across all matches ──────────────────────
from pathlib import Path

data_dir = Path(DATA_DIR) if 'DATA_DIR' in globals() else Path('../data/raw')
event_files = sorted(data_dir.glob('*_events.json'))

all_events_raw = []
for path in event_files:
    with open(path, errors='ignore') as f:
        content = f.read()

    try:
        events = json.loads(content)
    except json.JSONDecodeError:
        events = []
        for match in re.finditer(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', content):
            try:
                events.append(json.loads(match.group()))
            except json.JSONDecodeError:
                pass

    if isinstance(events, dict):
        events = [events]

    all_events_raw.extend(events)

df_all_events = pd.json_normalize(all_events_raw)
event_counts = df_all_events['event.event_type_name'].value_counts()
total = len(df_all_events)

# Build a summary dataframe including percentage and cumulative percentage
df_event_summary = event_counts.rename_axis('event').reset_index(name='count')
df_event_summary['percent'] = df_event_summary['count'] / total * 100
df_event_summary['cumulative_percent'] = df_event_summary['percent'].cumsum()

print(f"Loaded {len(event_files)} event files")
print(f"Total events across all matches: {total:,}")
print(f"Unique event types across all matches: {len(event_counts)} (out of {len(df_event_types)} defined)\n")
# Display the summary table (counts, percent, cumulative_percent)
from IPython.display import display
display(df_event_summary)


Loaded 34 event files
Total events across all matches: 52,040
Unique event types across all matches: 27 (out of 84 defined)



,event,count,percent,cumulative_percent
0,PASS,34368,66.041507,66.041507
1,BALL TOUCH,4833,9.287087,75.328593
2,AERIAL,2174,4.177556,79.506149
3,TACKLE,2034,3.908532,83.414681
4,BALL RECOVERY,1433,2.753651,86.168332
5,FOUL,1433,2.753651,88.921983
6,TAKEON,1265,2.430822,91.352806
7,INTERCEPTION,724,1.391238,92.744043
8,CHALLENGE,667,1.281706,94.025749
9,DISPOSSESSED,608,1.168332,95.194081


**Key insights:**
- Event type coverage is measured across all available event files, not just one match.
- **Severe class imbalance**: PASS dominates the event distribution; use per-class metrics rather than accuracy.
- Rare-but-valuable events (GOAL, SAVED SHOT, MISSED SHOT, CARD) each make up <1% of events — accuracy would be a meaningless metric, per-class F1 is required.


## Part 2 - Events + Tracking: Coordinate System and Attacking Direction
**Goal:** validate that event coordinates are already on the 0-100 attacking-direction scale, then convert tracking ball/player x-y coordinates from physical meters to the same 0-100 field scale for comparison. Step 2 keeps both coordinate systems raw. The training-table feature step intentionally excludes event coordinate columns to avoid leakage and keeps only a period-based attacking-direction flag.


In [6]:
# Cell 3: Coordinate system analysis

from IPython.display import display

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

# Use all loaded event files, not just the displayed match.
events_for_coords = df_all_events.copy() if 'df_all_events' in globals() else df_events.copy()
if 'df_all_events' not in globals():
    print('df_all_events not found; using df_events only. Run Cell 2 first to include every match.')

coord_cols = [c for c in ['x', 'y', 'x_start', 'y_start', 'x_end', 'y_end'] if c in events_for_coords.columns]
events_for_coords[coord_cols] = events_for_coords[coord_cols].apply(pd.to_numeric, errors='coerce')

# Events are already normalized 0-100 by the provider. Do not rescale them here.
df_all_events_checked = events_for_coords.copy()

# Normalize tracking x/y coordinates from meters to the same 0-100 field scale.
tracking_ball_rows = []
tracking_player_rows = []
for frame in tracking_frames:
    ball = frame.get('ball') or [None, None, None]
    clock = frame.get('match_clock') or [None, None]
    ball_x_raw = pd.to_numeric(ball[0], errors='coerce')
    ball_y_raw = pd.to_numeric(ball[1], errors='coerce')
    tracking_ball_rows.append({
        'frame_id': frame.get('frame'),
        'period_id': frame.get('period'),
        'match_clock_min': clock[0],
        'match_clock_sec': clock[1],
        'ball_x_raw_m': ball[0],
        'ball_y_raw_m': ball[1],
        'ball_z_raw_m': ball[2],
        'ball_x_norm': min(100, max(0, ball_x_raw / PITCH_LENGTH_M * 100)) if pd.notna(ball_x_raw) else pd.NA,
        'ball_y_norm': min(100, max(0, ball_y_raw / PITCH_WIDTH_M * 100)) if pd.notna(ball_y_raw) else pd.NA,
    })

    for team_id, players in (frame.get('data') or {}).items():
        for player in players:
            player_x_raw = pd.to_numeric(player.get('x'), errors='coerce')
            player_y_raw = pd.to_numeric(player.get('y'), errors='coerce')
            tracking_player_rows.append({
                'frame_id': frame.get('frame'),
                'period_id': frame.get('period'),
                'match_clock_min': clock[0],
                'match_clock_sec': clock[1],
                'team_id': team_id,
                'player_id': player.get('id'),
                'player_visible': player.get('vis'),
                'player_x_raw_m': player.get('x'),
                'player_y_raw_m': player.get('y'),
                'player_x_norm': min(100, max(0, player_x_raw / PITCH_LENGTH_M * 100)) if pd.notna(player_x_raw) else pd.NA,
                'player_y_norm': min(100, max(0, player_y_raw / PITCH_WIDTH_M * 100)) if pd.notna(player_y_raw) else pd.NA,
            })

df_tracking_ball_normalized = pd.DataFrame(tracking_ball_rows)
df_tracking_players_normalized = pd.DataFrame(tracking_player_rows)

# Events coordinate range across every loaded match.
match_count = df_all_events_checked['match_id'].nunique() if 'match_id' in df_all_events_checked.columns else 1
print('=== EVENTS coordinate system (all matches) ===')
print(f'  Matches: {match_count}')
for col in coord_cols:
    values = df_all_events_checked[col].dropna()
    out_of_range = values.lt(0).sum() + values.gt(100).sum()
    print(f'  {col:<7} range: {values.min()} - {values.max()} | outside 0-100: {out_of_range}')
print('  Events are already normalized 0-100; ETL does not rescale them')

# Do shots cluster at high X? This checks attacking-direction normalization across all matches.
shot_types = ['GOAL', 'SAVED SHOT', 'MISSED SHOT', 'SHOT ON POST']
shots = df_all_events_checked[df_all_events_checked['event.event_type_name'].isin(shot_types)].copy()
shot_summary = (
    shots
    .groupby(['match_id', 'team.team_name', 'period_id'], dropna=False)
    .agg(
        shots=('event.event_type_name', 'size'),
        x_median=('x', 'median'),
        x_min=('x', 'min'),
        x_max=('x', 'max'),
        y_median=('y', 'median'),
    )
    .reset_index()
    .sort_values(['match_id', 'team.team_name', 'period_id'])
)

print()
print('=== SHOT LOCATIONS SUMMARY (all matches, both teams, both halves) ===')
display(shot_summary)

print()
print('=== TRACKING coordinate system (match 679026) ===')
print('Ball coordinates:')
print(df_tracking_ball_normalized[['ball_x_raw_m', 'ball_y_raw_m', 'ball_x_norm', 'ball_y_norm']].describe())
print()
print('Player coordinates:')
print(df_tracking_players_normalized[['player_x_raw_m', 'player_y_raw_m', 'player_x_norm', 'player_y_norm']].describe())

print()
print('Sample normalized tracking ball rows:')
display(df_tracking_ball_normalized.head())
print()
print('Sample normalized tracking player rows:')
display(df_tracking_players_normalized.head())

print()
print('=== SHARED FIELD COORDINATE DECISION ===')
print('  Events:   already 0-100, always attacking toward 100')
print('  Tracking: physical meters on a 105x68 pitch')
print('  ETL:      normalize tracking x/y only, for comparison and modelling')
print('  Features: convert primary event x/y to absolute meters and add attacking direction')


=== EVENTS coordinate system (all matches) ===
  Matches: 33
  x       range: 0.0 - 100.0 | outside 0-100: 0
  y       range: 0.0 - 100.0 | outside 0-100: 0
  x_start range: 0.0 - 100.0 | outside 0-100: 0
  y_start range: 0.0 - 100.0 | outside 0-100: 0
  x_end   range: 0.0 - 100.0 | outside 0-100: 0
  y_end   range: 0.0 - 100.0 | outside 0-100: 0
  Events are already normalized 0-100; ETL does not rescale them

=== SHOT LOCATIONS SUMMARY (all matches, both teams, both halves) ===


,match_id,team.team_name,period_id,shots,x_median,x_min,x_max,y_median
0,678949,Manchester United,1,9,86.30,80.7,95.3,47.80
1,678949,Manchester United,2,6,85.40,77.2,92.0,44.30
2,678949,Sunderland,1,3,91.50,75.9,94.8,49.10
3,678949,Sunderland,2,4,83.75,78.4,88.4,56.20
4,679026,Arsenal FC,1,8,88.20,73.5,94.3,41.75
...,...,...,...,...,...,...,...,...
124,714000,FK Bodo/Glimt,2,8,92.00,82.0,96.8,54.15
125,745399,Atalanta,1,2,91.00,90.4,91.6,55.20
126,745399,Atalanta,2,4,78.65,73.5,87.3,52.40
127,745399,Borussia Dortmund,1,7,86.30,70.5,95.9,52.50



=== TRACKING coordinate system (match 679026) ===
Ball coordinates:
       ball_x_raw_m  ball_y_raw_m
count  31630.000000  31630.000000
mean      49.863974     46.808958
std       23.517833     22.922541
min      -43.530000     -6.230000
25%       32.880000     29.240000
50%       50.405000     45.930000
75%       66.950000     66.220000
max      265.540000    205.790000

Player coordinates:
       player_x_raw_m  player_y_raw_m  player_x_norm  player_y_norm
count    1.309044e+06    1.309044e+06   1.309044e+06   1.309044e+06
mean     5.459046e+01    3.447023e+01   5.199082e+01   5.069109e+01
std      2.562294e+01    1.392535e+01   2.440237e+01   2.047610e+01
min     -1.770000e+00   -1.940000e+00   0.000000e+00   0.000000e+00
25%      3.566000e+01    2.548000e+01   3.396190e+01   3.747059e+01
50%      5.519000e+01    3.423000e+01   5.256190e+01   5.033824e+01
75%      7.425000e+01    4.325000e+01   7.071429e+01   6.360294e+01
max      1.071300e+02    7.099000e+01   1.000000e+02   1.000

,frame_id,period_id,match_clock_min,match_clock_sec,ball_x_raw_m,ball_y_raw_m,ball_z_raw_m,ball_x_norm,ball_y_norm
0,0,1,0,1,NaN,NaN,NaN,<NA>,<NA>
1,1,1,0,1,NaN,NaN,NaN,<NA>,<NA>
2,2,1,0,1,47.21,47.87,0.925,44.961905,70.397059
3,3,1,0,1,46.98,48.22,0.556,44.742857,70.911765
4,4,1,0,1,46.74,48.58,0.187,44.514286,71.441176



Sample normalized tracking player rows:


,frame_id,period_id,match_clock_min,match_clock_sec,team_id,player_id,player_visible,player_x_raw_m,player_y_raw_m,player_x_norm,player_y_norm
0,0,1,0,1,8561,1553280,False,48.88,61.48,46.552381,90.411765
1,0,1,0,1,8561,1161475,False,54.64,9.92,52.038095,14.588235
2,0,1,0,1,8561,1221904,False,55.40,27.14,52.761905,39.911765
3,0,1,0,1,8561,1170336,False,66.40,38.27,63.238095,56.279412
4,0,1,0,1,8561,1329572,False,52.08,21.28,49.600000,31.294118



=== SHARED FIELD COORDINATE DECISION ===
  Events:   already 0-100, always attacking toward 100
  Tracking: physical meters on a 105x68 pitch
  ETL:      normalize tracking x/y only, for comparison and modelling
  Features: convert primary event x/y to absolute meters and add attacking direction


**Key insights:**
- Events already use a **normalized 0-100 coordinate system** for x/y, start x/y, and end x/y across all loaded event files. We validate this; we do not rescale event coordinates.
- Shots cluster at high x across teams, periods, and matches, confirming events are normalized to "always attacking toward x=100", independent of the physical pitch side.
- Tracking starts in **physical meters on a 105x68 pitch**, so ETL converts and clips tracking ball/player x-y into the same **0-100 field scale**.
- Step 2 keeps event and tracking coordinates in their raw source systems. The training-table feature step excludes event coordinate columns to avoid leakage and adds an explicit period-based attacking-direction flag. Speeds and distances remain in meters because those are physical features.


## Part 3 — Events File: Time Encoding, Outcome & Qualifiers
**Goal:** check how time is encoded (does the minute reset at half-time?) and compare it to the tracking file's `match_clock`, check what the `outcome` field means across event types, and inspect what's inside the `qualifiers` field.


In [7]:
# ── Cell 4: Time encoding analysis ───────────────────────────────────────────

# Events time encoding
print("=== EVENTS time encoding ===")
p1 = df_events[df_events['period_id'] == 1]
p2 = df_events[df_events['period_id'] == 2]
print(f"  Period 1 minutes: {p1['min'].min()} - {p1['min'].max()}")
print(f"  Period 2 minutes: {p2['min'].min()} - {p2['min'].max()}")
print(f"  → Minutes count UP and do NOT reset at half-time")

# Tracking time encoding
print(f"\n=== TRACKING time encoding ===")
p1_frames = [f for f in tracking_frames if f['period'] == 1]
p2_frames = [f for f in tracking_frames if f['period'] == 2]
p1_times = [f['match_clock'] for f in p1_frames]
p2_times = [f['match_clock'] for f in p2_frames]
print(f"  Period 1 frames:      {len(p1_frames)}")
print(f"  Period 2 frames:      {len(p2_frames)}")
print(f"  Period 1 match_clock: {p1_times[0]} → {p1_times[-1]}")
print(f"  Period 2 match_clock: {p2_times[0]} → {p2_times[-1]}")

print(f"\n=== TIME SYNC CHALLENGE ===")
print(f"  Events use:   period + min + sec + millisec")
print(f"  Tracking uses: period + match_clock [min, sec] + frame")
print(f"  → match_clock[0]=min, match_clock[1]=sec — compatible!")
print(f"  → But millisec drift may still exist — needs verification")

# ── Outcome field meaning ─────────────────────────────────────────────────────
print(f"\n=== OUTCOME FIELD ===")
outcome_by_type = df_events.groupby('event.event_type_name')['outcome'].unique()
always_true = [name for name, vals in outcome_by_type.items() if set(vals) == {True}]
varies = [name for name, vals in outcome_by_type.items() if len(set(vals)) > 1]
print(f"  Always True (not informative):  {always_true}")
print(f"  Varies True/False (success/fail): {varies}")

# ── Qualifiers structure ──────────────────────────────────────────────────────
print(f"\n=== QUALIFIERS STRUCTURE ===")
sample_quals = next(q for q in df_events['qualifiers'] if isinstance(q, list) and len(q) > 0)
print(f"  Example qualifiers list (one event):")
for q in sample_quals:
    print(f"    {q}")


=== EVENTS time encoding ===
  Period 1 minutes: 0 - 47
  Period 2 minutes: 45 - 96
  → Minutes count UP and do NOT reset at half-time

=== TRACKING time encoding ===
  Period 1 frames:      28980
  Period 2 frames:      30522
  Period 1 match_clock: [0, 1] → [48, 4]
  Period 2 match_clock: [48, 5] → [99, 6]

=== TIME SYNC CHALLENGE ===
  Events use:   period + min + sec + millisec
  Tracking uses: period + match_clock [min, sec] + frame
  → match_clock[0]=min, match_clock[1]=sec — compatible!
  → But millisec drift may still exist — needs verification

=== OUTCOME FIELD ===
  Always True (not informative):  ['BALL RECOVERY', 'BALL TOUCH', 'CARD', 'CLEARANCE', 'DISPOSSESSED', 'END', 'GOAL', 'INTERCEPTION', 'MISSED SHOT', 'SAVE', 'SAVED SHOT', 'SUBSTITUTION OFF', 'SUBSTITUTION ON']
  Varies True/False (success/fail): ['AERIAL', 'CLAIM', 'FOUL', 'PASS', 'TACKLE', 'TAKEON']

=== QUALIFIERS STRUCTURE ===
  Example qualifiers list (one event):
    {'qualifier_id': 140, 'qualifier_name': 'Pa

**Key insights:**
- Event minutes **count up continuously** and do **not** reset at half-time (period 2 starts ~minute 45) — same convention as tracking's `match_clock`, so the two should sync on `[period, min, sec]`. Millisecond-level drift still needs checking once frame-level alignment starts (Step 2).
- `outcome` does **not** mean the same thing for every event type: for BALL TOUCH, BALL RECOVERY and INTERCEPTION it is always `True` (not informative); for PASS, AERIAL, FOUL, TAKEON and TACKLE it varies and represents success/failure.
- `qualifiers` is a list of `{qualifier_id, qualifier_name, value}` dicts with extra event-specific detail (e.g. pass end x/y). It's useful context, not a separate set of events to detect.


## Part 4 — Tracking File: Data Quality & Sampling Rate
**Goal:** check the ball's presence rate, how much player tracking is observed (`vis=True`) vs AI-imputed (`vis=False`), what the `cam` field tells us about reliable live play, and confirm the tracking sampling rate.


In [8]:
# ── Cell 5: Ball and visibility quality ──────────────────────────────────────

total = len(tracking_frames)

# Ball presence
ball_present = sum(1 for f in tracking_frames if any(v is not None for v in f['ball']))
ball_missing = total - ball_present

# Cam presence (reliable live play)
cam_present = sum(1 for f in tracking_frames if f.get('cam') is not None)

# vis=True vs False
vis_true = vis_false = 0
for f in tracking_frames:
    for team_key in f['data']:
        for p in f['data'][team_key]:
            if p.get('vis'): vis_true += 1
            else: vis_false += 1

# Ball during live play only (cam present)
live_frames = [f for f in tracking_frames if f.get('cam') is not None]
ball_in_live = sum(1 for f in live_frames if any(v is not None for v in f['ball']))

print("=== BALL PRESENCE ===")
print(f"  All frames:        {ball_present:>6} / {total}  ({ball_present/total*100:.1f}%)")
print(f"  Missing:           {ball_missing:>6} / {total}  ({ball_missing/total*100:.1f}%)")
print(f"  During live play:  {ball_in_live:>6} / {cam_present}  ({ball_in_live/cam_present*100:.1f}%)")

print(f"\n=== PLAYER VISIBILITY ===")
total_vis = vis_true + vis_false
print(f"  vis=True  (observed):  {vis_true:>7}  ({vis_true/total_vis*100:.1f}%)")
print(f"  vis=False (imputed):   {vis_false:>7}  ({vis_false/total_vis*100:.1f}%)")

print(f"\n=== CAM FIELD ===")
print(f"  Frames with cam:    {cam_present:>6} / {total}  ({cam_present/total*100:.1f}%)")
print(f"  → Only use cam-present frames for reliable detection")

print(f"\n=== KEY WARNINGS ===")
print(f"  ⚠️  Ball missing in {ball_missing/total*100:.1f}% of all frames")
print(f"  ⚠️  {vis_false/total_vis*100:.1f}% of player positions are AI-imputed, not observed")
print(f"  ⚠️  Ball can be missing exactly during shots and tackles — do not trust blindly")

# ── Sampling rate ──────────────────────────────────────────────────────────────
print(f"\n=== SAMPLING RATE ===")
p1_frames_chk = [f for f in tracking_frames if f['period'] == 1]
start_clock, end_clock = p1_frames_chk[0]['match_clock'], p1_frames_chk[-1]['match_clock']
duration_sec = (end_clock[0]*60 + end_clock[1]) - (start_clock[0]*60 + start_clock[1])
fps = len(p1_frames_chk) / duration_sec
print(f"  Period 1: {len(p1_frames_chk)} frames over ~{duration_sec}s → ~{fps:.1f} Hz")
print(f"  → Confirms ~10 frames/second, as documented")


=== BALL PRESENCE ===
  All frames:         31630 / 59502  (53.2%)
  Missing:            27872 / 59502  (46.8%)
  During live play:   31354 / 34252  (91.5%)

=== PLAYER VISIBILITY ===
  vis=True  (observed):   467800  (35.7%)
  vis=False (imputed):    841244  (64.3%)

=== CAM FIELD ===
  Frames with cam:     34252 / 59502  (57.6%)
  → Only use cam-present frames for reliable detection

=== KEY WARNINGS ===
  ⚠️  Ball missing in 46.8% of all frames
  ⚠️  64.3% of player positions are AI-imputed, not observed
  ⚠️  Ball can be missing exactly during shots and tackles — do not trust blindly

=== SAMPLING RATE ===
  Period 1: 28980 frames over ~2883s → ~10.1 Hz
  → Confirms ~10 frames/second, as documented


**Key insights:**
- Tracking runs at **~10 frames/second**, as documented.
- The ball is missing in **47%** of all frames, but only **8.5%** during reliable live play (`cam` present) — still enough to cause issues exactly during fast events like shots and tackles.
- **64%** of player positions are AI-imputed (`vis=False`), not directly observed — treat off-ball positions with some caution.
- Only **58%** of frames have `cam` present — restrict detection logic to these frames for reliability.


## Part 5 — Cross-File Consistency & Data Asset Inventory
**Goal:** verify that team and player IDs match between the events and tracking files for match 679026, sanity-check time alignment, and check which matches in `DATA/` actually have *both* an events file and a tracking file (modelling needs both).


In [9]:
# ── Cell 6: Cross-file consistency (events ↔ tracking) + asset inventory ────
# Both files are now from match 679026

# ── Data asset inventory ───────────────────────────────────────────────────────
import re as _re
print("=== DATA ASSET INVENTORY ===")
matches = {}
for fn in sorted(os.listdir(DATA_DIR)):
    m = _re.match(r'(\d+)_(events|tracking_data)\.', fn)
    if m:
        matches.setdefault(m.group(1), set()).add(m.group(2))
for match_id, assets in matches.items():
    has_events = 'events' in assets
    has_tracking = 'tracking_data' in assets
    print(f"  Match {match_id}: events={'✅' if has_events else '❌'}  tracking={'✅' if has_tracking else '❌'}")

# Team IDs in tracking
tracking_team_ids = set(tracking_header['players_data'].keys())
print("=== TRACKING team IDs ===")
for team_id in tracking_team_ids:
    players = tracking_header['players_data'][team_id]
    names = [p['name'] for p in players.values()][:3]
    print(f"  team_id: {team_id}  ({len(players)} players)  e.g. {names}")

# Team IDs in events
event_teams = df_events[['team.team_id','team.team_name']].drop_duplicates()
print(f"\n=== EVENTS team IDs ===")
for _, row in event_teams.iterrows():
    print(f"  team_id: {row['team.team_id']}  ({row['team.team_name']})")

# Player ID overlap
tracking_player_ids = set()
for team_id in tracking_header['players_data']:
    for pid in tracking_header['players_data'][team_id].keys():
        tracking_player_ids.add(str(pid))
event_player_ids = set(df_events['player.player_id'].astype(str).unique())
overlap = event_player_ids & tracking_player_ids

print(f"\n=== PLAYER ID OVERLAP ===")
print(f"  Players in tracking: {len(tracking_player_ids)}")
print(f"  Players in events:   {len(event_player_ids)}")
print(f"  Overlap:             {len(overlap)}")
if len(overlap) > 0:
    print(f"  ✅ IDs match — same match confirmed")
else:
    print(f"  ⚠️  No overlap — check if files are from the same match")

# Time alignment sample
print(f"\n=== TIME ALIGNMENT SAMPLE ===")
for _, row in df_events.head(5).iterrows():
    print(f"  Event: {row['event.event_type_name']:<15} P{row['period_id']} {int(row['min']):02d}:{int(row['sec']):02d}  →  tracking match_clock=[{int(row['min'])},{int(row['sec'])}]")

=== DATA ASSET INVENTORY ===
  Match 678949: events=✅  tracking=✅
  Match 679026: events=✅  tracking=✅
  Match 679053: events=✅  tracking=✅
  Match 679072: events=✅  tracking=✅
  Match 679075: events=✅  tracking=✅
  Match 679088: events=✅  tracking=✅
  Match 679104: events=✅  tracking=✅
  Match 679128: events=✅  tracking=✅
  Match 682607: events=✅  tracking=✅
  Match 682717: events=✅  tracking=✅
  Match 682815: events=✅  tracking=✅
  Match 683132: events=✅  tracking=✅
  Match 683190: events=✅  tracking=✅
  Match 683231: events=✅  tracking=❌
  Match 683253: events=✅  tracking=✅
  Match 683261: events=✅  tracking=✅
  Match 683309: events=✅  tracking=✅
  Match 683425: events=✅  tracking=✅
  Match 684014: events=✅  tracking=✅
  Match 684119: events=✅  tracking=✅
  Match 684139: events=✅  tracking=✅
  Match 684141: events=✅  tracking=✅
  Match 684147: events=✅  tracking=✅
  Match 689340: events=✅  tracking=✅
  Match 689526: events=✅  tracking=✅
  Match 689552: events=✅  tracking=✅
  Match 6

**Key insights:**
- Team IDs and 31/31 player IDs match between the events and tracking files for match 679026 — confirmed same match.
- The first few events line up correctly with tracking's `match_clock`, supporting the time-sync approach from Part 3.
- Of the 3 matches in `DATA/`, only **679026** has both an events file and a tracking file. `678949` has tracking only, and `683231` (FC Barcelona, different league) has events only — so **679026 is currently our only match for the alignment pipeline (Step 2 onward)**.


## Summary - Key Takeaways for the Team
1. **Two coordinate systems**: events are already normalized 0-100 and always attack toward 100; tracking is physical meters on a 105x68 pitch, with teams switching ends at half-time. Step 2 keeps both source systems raw, and the training-table feature step excludes event coordinate columns from the pass-detector table to avoid leakage.
2. **Two time references**: both count up continuously across both halves and look directly compatible on `[period, min, sec]`, but frame-level joining still needs match-clock interpolation at 10 Hz.
3. **The ball is the key signal but incomplete**: present in ~91.5% of live-play frames, never imputed when missing, so gaps need explicit handling.
4. **~64% of player positions are AI-imputed**, not observed, so off-ball features should be treated carefully.
5. **Severe class imbalance**: PASS dominates the event distribution, so use per-class F1 rather than accuracy.
6. **Use all matched event/tracking files for modelling**: Step 2 builds one all-match master join table from every match that has both raw files.

**Next step:** Step 2 - join events to tracking by match clock, filter to `cam`-present frames, interpolate short ball gaps, and keep source tracking/event fields intact for feature building.
